In [1]:
import pandas as pd
import kagglehub
import ast
import numpy as np
import glob

In [2]:
# Download MovieLens and TMDB datasets
path_ml = kagglehub.dataset_download("grouplens/movielens-latest-full")
print("MovieLens path:", path_ml)

path_tmdb = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("TMDB path:", path_tmdb)

MovieLens path: /Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1
TMDB path: /Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [3]:
glob.glob(f"{path_ml}/*")

['/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-tags.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/README.md',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-scores.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv']

In [4]:
glob.glob(f"{path_tmdb}/*")


['/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links_small.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/ratings.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/ratings_small.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv']

In [5]:
ratings_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')
tags_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv')

# Aggregate tags per movie
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()


In [6]:
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')
links_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links.csv')


In [7]:
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']


In [8]:
# Clean links
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Clean metadata
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)
metadata_tmdb['main_genre'] = metadata_tmdb['genres'].apply(lambda x: x[0]['name'] if isinstance(x, list) and x else None)


In [9]:
# Merge MovieLens movies with TMDB metadata
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')


In [10]:
# Parse JSON columns
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)

# Helper functions
def get_director(crew):
    for person in crew:
        if person.get('job') == 'Director':
            return person.get('name')
    return None

def get_lead_actor(cast):
    return cast[0]['name'] if isinstance(cast, list) and cast else None

# Extract info
credits_tmdb['tmdbId'] = credits_tmdb['id']
credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)

# Merge
movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'lead_actor']], on='tmdbId', how='left')


In [11]:
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')


In [12]:
# Merge tags
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')

# Merge rating statistics
movies_full = pd.merge(movies_full, rating_stats, on='movieId', how='left')


In [13]:
# Extract year
movies_full['release_year'] = pd.to_datetime(movies_full['release_date'], errors='coerce').dt.year

# Create runtime bins (30-min intervals)
bin_edges = list(range(0, 301, 30)) + [np.inf]
labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+'
          for i in range(len(bin_edges) - 1)]

movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)


In [14]:
movies_full

,movieId,title_x,genres_x,imdbId,tmdbId,adult,belongs_to_collection,budget,genres_y,homepage,...,director,lead_actor,keywords,tag,vote_average_y,vote_min,vote_max,vote_count_y,release_year,runtime_bin
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,...,John Lasseter,Tom Hanks,"[jealousy, toy, boy, friendship, friends, riva...","[toy comes to life, funny, exciting plot, want...",3.886649,0.5,5.0,68469.0,1995.0,60–90min
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,...,Joe Johnston,Robin Williams,"[board game, disappearance, based on children'...","[family, based on a book, Lebbat, Dynamic CGI ...",3.246583,0.5,5.0,27143.0,1995.0,90–120min
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,...,Howard Deutch,Walter Matthau,"[fishing, best friend, duringcreditsstinger, o...","[funny, Daryl Hannah, Funniest Movies, fishing...",3.173981,0.5,5.0,15585.0,1995.0,90–120min
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,...,Forest Whitaker,Whitney Houston,"[based on novel, interracial relationship, sin...","[characters, chick flick, interracial relation...",2.874540,0.5,5.0,2989.0,1995.0,120–150min
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,...,Charles Shyer,Steve Martin,"[baby, midlife crisis, confidence, aging, daug...","[family, gynecologist, sequel fever, CLV, preg...",3.077291,0.5,5.0,15474.0,1995.0,90–120min
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46893,181393,Venice (2010),Drama|Romance,1684935,79782,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,...,Jan Jakub Kolski,Marcin Walewski,[],NaN,3.500000,3.5,3.5,1.0,2010.0,90–120min
46894,181751,Lagaan: Once Upon a Time in India (2001),Adventure|Drama|Romance,169102,19666,False,NaN,5200000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",http://www.lagaan.com,...,Ashutosh Gowariker,Aamir Khan,"[sport, british, bollywood, arrogance, based o...","[bollywood, sport, arrogance, 19th century, dr...",3.980769,2.5,5.0,26.0,2001.0,210–240min
46895,183123,Taboo (2002),Drama|Horror|Mystery|Thriller,288243,97206,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 27, 'name...",NaN,...,Max Makowski,Nick Stahl,[],NaN,2.166667,1.0,3.0,3.0,2002.0,60–90min
46896,187127,David Lynch: The Art Life (2017),Documentary,1691152,413765,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,...,Olivia Neergaard-Holm,David Lynch,[],[Criterion],3.750000,3.0,5.0,6.0,2017.0,60–90min


In [15]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title_y', 'video', 'vote_average_x', 'vote_count_x',
       'main_genre', 'director', 'lead_actor', 'keywords', 'tag',
       'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'release_year', 'runtime_bin'],
      dtype='object')

In [16]:
movies_full["same_titme_xy"] = movies_full["title_x"] == movies_full["title_y"]

In [17]:
movies_full["same_titme_xy"].value_counts()

same_titme_xy
False    46746
True       152
Name: count, dtype: int64

In [18]:
# Extract year from MovieLens-style title_x (e.g., "Toy Story (1995)")
movies_full['year_from_title_x'] = movies_full['title_x'].str.extract(r'\((\d{4})\)').astype(float)

# Compare with TMDB's release_year
year_mismatch = (movies_full['year_from_title_x'] != movies_full['release_year']) & movies_full['release_year'].notnull()
print("Year mismatch count:", year_mismatch.sum())


Year mismatch count: 2277


In [19]:
movies_full['year_from_title_x']

0        1995.0
1        1995.0
2        1995.0
3        1995.0
4        1995.0
          ...  
46893    2010.0
46894    2001.0
46895    2002.0
46896    2017.0
46897    2008.0
Name: year_from_title_x, Length: 46898, dtype: float64

In [20]:
movies_full['release_year_tmdb'] = movies_full['release_year']
movies_full['release_year_ml'] = movies_full['year_from_title_x']


In [21]:
movies_full['year_mismatch'] = (
    movies_full['release_year'] != movies_full['year_from_title_x']
)

movies_full['year_mismatch'].value_counts()

year_mismatch
False    44533
True      2365
Name: count, dtype: int64

In [22]:
# Extract year from MovieLens title
movies_full['year_from_title_x'] = movies_full['title_x'].str.extract(r'\((\d{4})\)').astype(float)

# Flag mismatches between TMDB and MovieLens years
movies_full['year_mismatch'] = (movies_full['year_from_title_x'] != movies_full['release_year'])

# Rename TMDB year for clarity
movies_full = movies_full.rename(columns={'release_year': 'release_year_tmdb'})

# Optional: Keep only one title column (prefer TMDB)
# movies_full = movies_full.rename(columns={'title_y': 'title'})
# movies_full = movies_full.drop(columns=['title_x'])

# If needed: Keep both years and the mismatch flag
movies_full = movies_full.rename(columns={'year_from_title_x': 'release_year_ml'})


In [ ]:
# Parse release_date safely
movies_full['release_date_parsed'] = pd.to_datetime(movies_full['release_date'], errors='coerce')

# Extract release year from parsed release_date (as a Series)
release_year_tmdb = movies_full['release_date_parsed'].dt.year

# Extract release year from MovieLens title (as a Series)
release_year_ml = movies_full['title_x'].str.extract(r'\((\d{4})\)')[0].astype(float)

# Combine: use TMDB release year if available, else fall back to MovieLens
movies_full['release_year'] = release_year_tmdb.combine_first(release_year_ml)

movies_full['vote_count'] = movies_full[['vote_count_x', 'vote_count_y']].max(axis=1)
movies_full['vote_average'] = movies_full.apply(
    lambda row: row['vote_average_x'] if row['vote_count_x'] >= row['vote_count_y'] else row['vote_average_y'],
    axis=1
)
movies_full['release_year_merged'] = movies_full[['release_year_tmdb', 'release_year_ml']].min(axis=1)
movies_full['title'] = movies_full['title_y'].combine_first(movies_full['title_x'])

In [24]:
print(type(movies_full['release_year_tmdb']))  # Should be <class 'pandas.core.series.Series'>
print(movies_full[['release_year_tmdb']].head())  # This returns a DataFrame (note the double brackets!)


<class 'pandas.core.frame.DataFrame'>
   release_year_tmdb  release_year_tmdb
0             1995.0             1995.0
1             1995.0             1995.0
2             1995.0             1995.0
3             1995.0             1995.0
4             1995.0             1995.0


In [26]:
movies_full.loc[movies_full["release_year"].isna()]

,movieId,title_x,genres_x,imdbId,tmdbId,adult,belongs_to_collection,budget,genres_y,homepage,...,vote_count_y,release_year_tmdb,runtime_bin,same_titme_xy,release_year_ml,release_year_tmdb,release_year_ml,year_mismatch,release_date_parsed,release_year
27889,125958,Stephen Fry In America - New World,(no genres listed),1307789,76162,False,NaN,0,[],http://www.stephenfry.com/2008/10/10/stephen-f...,...,18.0,NaN,30–60min,True,NaN,NaN,NaN,True,NaT,NaN
31018,135691,Señorita Justice,(no genres listed),374212,201913,False,NaN,0,[],NaN,...,1.0,NaN,60–90min,True,NaN,NaN,NaN,True,NaT,NaN
31489,136880,Vaastupurush,(no genres listed),396963,91527,False,NaN,0,[],NaN,...,2.0,NaN,150–180min,True,NaN,NaN,NaN,True,NaT,NaN
31640,137292,L'uomo della carità,(no genres listed),1010054,213648,False,NaN,0,[],NaN,...,1.0,NaN,90–120min,True,NaN,NaN,NaN,True,NaT,NaN
31873,138120,The Expedition,(no genres listed),1457759,297173,False,NaN,0,[],NaN,...,1.0,NaN,90–120min,True,NaN,NaN,NaN,True,NaT,NaN
33149,141708,Disaster Playground,Documentary,3900822,324266,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,...,1.0,NaN,60–90min,True,NaN,NaN,NaN,True,NaT,NaN
34677,146010,Sentimentalnyy roman,(no genres listed),75190,263823,False,NaN,0,[],NaN,...,1.0,NaN,NaN,True,NaN,NaN,NaN,True,NaT,NaN
34712,146082,Yedyanchi Jatra,Children|Comedy|Drama,2290845,100783,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,...,1.0,NaN,120–150min,True,NaN,NaN,NaN,True,NaT,NaN
34872,146429,Monk by Blood,(no genres listed),3631388,262113,False,NaN,0,[],NaN,...,1.0,NaN,0–30min,True,NaN,NaN,NaN,True,NaT,NaN
34882,146463,I Am Syd Stone,(no genres listed),3493688,366763,False,NaN,0,[],https://www.peccapics.com/product/boys-on-film...,...,2.0,NaN,0–30min,True,NaN,NaN,NaN,True,NaT,NaN


In [28]:
movies_full.columns

Index(['movieId', 'title_x', 'genres_x', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_y', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title_y', 'video', 'vote_average_x', 'vote_count_x',
       'main_genre', 'director', 'lead_actor', 'keywords', 'tag',
       'vote_average_y', 'vote_min', 'vote_max', 'vote_count_y',
       'release_year_tmdb', 'runtime_bin', 'same_titme_xy', 'release_year_ml',
       'release_year_tmdb', 'release_year_ml', 'year_mismatch',
       'release_date_parsed', 'release_year'],
      dtype='object')

In [31]:
movies_full.loc[movies_full["vote_count_x"] != movies_full["vote_count_y"]][["vote_count_x","vote_count_y"]]

,vote_count_x,vote_count_y
0,5415.0,68469.0
1,2413.0,27143.0
2,92.0,15585.0
3,34.0,2989.0
4,173.0,15474.0
...,...,...
46893,4.0,1.0
46894,125.0,26.0
46895,9.0,3.0
46896,32.0,6.0


In [36]:
movies_full.loc[movies_full["title_x"] != movies_full["title_y"]][["title_x","title_y"]]

,title_x,title_y
0,Toy Story (1995),Toy Story
1,Jumanji (1995),Jumanji
2,Grumpier Old Men (1995),Grumpier Old Men
3,Waiting to Exhale (1995),Waiting to Exhale
4,Father of the Bride Part II (1995),Father of the Bride Part II
...,...,...
46893,Venice (2010),Venice
46894,Lagaan: Once Upon a Time in India (2001),Lagaan: Once Upon a Time in India
46895,Taboo (2002),Taboo
46896,David Lynch: The Art Life (2017),David Lynch: The Art Life


In [39]:
movies_full.loc[movies_full["title_x"] != movies_full["title_y"]][["title_x","title_y"]]["title_x"].str[-6:].unique()

array(['(1995)', '(1994)', '(1996)', '(1976)', '(1992)', '(1967)',
       '(1993)', '(1964)', '(1977)', '(1965)', '(1982)', '(1985)',
       '(1990)', '(1991)', '(1989)', '(1937)', '(1940)', '(1969)',
       '(1981)', '(1973)', '(1970)', '(1960)', '(1955)', '(1959)',
       '(1968)', '(1980)', '(1988)', '(1975)', '(1986)', '(1948)',
       '(1943)', '(1950)', '(1987)', '(1997)', '(1974)', '(1956)',
       '(1958)', '(1949)', '(1972)', '(1998)', '(1933)', '(1952)',
       '(1951)', '(1957)', '(1961)', '(1954)', '(1934)', '(1944)',
       '(1963)', '(1942)', '(1941)', '(1953)', '(1939)', '(1947)',
       '(1946)', '(1945)', '(1938)', '(1935)', '(1936)', '(1926)',
       '(1932)', '(1979)', '(1971)', '(1978)', '(1966)', '(1962)',
       '(1983)', '(1984)', '(1931)', '(1922)', '(1999)', '(1927)',
       '(1929)', '(1930)', '(1928)', '(1925)', '(2000)', '(1919)',
       '(1923)', '(1920)', '(1918)', '(1921)', '(2001)', '(1924)',
       '(2002)', '(2003)', '(1915)', '(2004)', '(1916)', '(191

In [41]:
movies_full.loc[movies_full["title_x"].str.endswith('Boys 3')][["title_x","title_y"]]

,title_x,title_y
30811,Bad Boys 3,Bad Boys for Life


In [43]:
movies_full.loc[movies_full["title_x"] != movies_full["title_y"]][["title_x","title_y"]]["title_y"].str[-6:].value_counts()


title_y
 Story    250
 Movie    209
 Night    168
 World    167
 House    108
         ... 
oquake      1
 Romae      1
Pastry      1
. Evil      1
eerama      1
Name: count, Length: 20737, dtype: int64

In [46]:
movies_full

,movieId,title_x,genres_x,imdbId,tmdbId,adult,belongs_to_collection,budget,genres_y,homepage,...,release_year_ml,release_year_tmdb,release_year_ml,year_mismatch,release_date_parsed,release_year,vote_count,vote_average,release_year_merged,title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,...,1995.0,1995.0,1995.0,False,1995-10-30,1995.0,68469.0,3.886649,1995.0,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,...,1995.0,1995.0,1995.0,False,1995-12-15,1995.0,27143.0,3.246583,1995.0,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,...,1995.0,1995.0,1995.0,False,1995-12-22,1995.0,15585.0,3.173981,1995.0,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,...,1995.0,1995.0,1995.0,False,1995-12-22,1995.0,2989.0,2.874540,1995.0,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,...,1995.0,1995.0,1995.0,False,1995-02-10,1995.0,15474.0,3.077291,1995.0,Father of the Bride Part II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46893,181393,Venice (2010),Drama|Romance,1684935,79782,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,...,2010.0,2010.0,2010.0,False,2010-05-25,2010.0,4.0,7.500000,2010.0,Venice
46894,181751,Lagaan: Once Upon a Time in India (2001),Adventure|Drama|Romance,169102,19666,False,NaN,5200000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",http://www.lagaan.com,...,2001.0,2001.0,2001.0,False,2001-06-15,2001.0,125.0,7.200000,2001.0,Lagaan: Once Upon a Time in India
46895,183123,Taboo (2002),Drama|Horror|Mystery|Thriller,288243,97206,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 27, 'name...",NaN,...,2002.0,2002.0,2002.0,False,2002-01-14,2002.0,9.0,2.900000,2002.0,Taboo
46896,187127,David Lynch: The Art Life (2017),Documentary,1691152,413765,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,...,2017.0,2017.0,2017.0,False,2017-02-15,2017.0,32.0,7.200000,2017.0,David Lynch: The Art Life
